# Real-Time Object Counting with YOLO + ByteTrack

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run each cell in order (Shift+Enter).

How it works:
1. **YOLO** detects objects in every frame
2. **ByteTrack** gives each object a stable ID across frames
3. **LineZone** adds +1 when an ID crosses the line

In [ ]:
# Step 1: install the libraries
!pip install -q -U ultralytics supervision trackers

In [ ]:
# Step 2: get a test video (free sample of cars on a highway)
from supervision.assets import download_assets, VideoAssets

SOURCE_VIDEO = download_assets(VideoAssets.VEHICLES)
TARGET_VIDEO = "counted_output.mp4"
print("Video ready:", SOURCE_VIDEO)

# Want your own video instead? Upload it in the Files panel (left)
# and write: SOURCE_VIDEO = "my_video.mp4"

In [ ]:
# Step 3: detect + track + count
import numpy as np
import supervision as sv
from ultralytics import YOLO
from trackers import ByteTrackTracker

model = YOLO("yolo11m.pt")   # "m" = medium: much more accurate than "n" (nano). Pretrained on 80 classes

# Which classes to count. COCO ids: 0=person, 2=car, 3=motorcycle, 5=bus, 7=truck, 39=bottle
CLASSES_TO_COUNT = [2, 3, 5, 7]

info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)
print(info)

# Horizontal counting line at 60% of the video height. Move it if needed.
y = int(info.height * 0.6)
line_zone = sv.LineZone(
    start=sv.Point(0, y), end=sv.Point(info.width, y),
    triggering_anchors=[sv.Position.CENTER],   # count when the box CENTER crosses
)

tracker = ByteTrackTracker(
    frame_rate=info.fps,
    track_activation_threshold=0.25,   # default 0.7 ignores less-confident objects
    high_conf_det_threshold=0.25,
    minimum_consecutive_frames=1,
    lost_track_buffer=60,
)
box_annotator = sv.BoxAnnotator(thickness=4)
label_annotator = sv.LabelAnnotator(text_scale=1.2, text_thickness=2)
line_annotator = sv.LineZoneAnnotator(thickness=4, text_scale=2, text_thickness=4)

def process_frame(frame, index):
    result = model(frame, imgsz=1280, conf=0.2, verbose=False)[0]   # 1280 helps with small/far objects
    detections = sv.Detections.from_ultralytics(result)
    detections = detections[np.isin(detections.class_id, CLASSES_TO_COUNT)]
    detections = tracker.update(detections)
    detections = detections[detections.tracker_id != -1]   # keep only confirmed tracks

    line_zone.trigger(detections)   # updates the count when an ID crosses the line

    labels = [f"#{tid}" for tid in detections.tracker_id]
    frame = box_annotator.annotate(frame.copy(), detections)
    frame = label_annotator.annotate(frame, detections, labels)
    frame = line_annotator.annotate(frame, line_counter=line_zone)
    return frame

sv.process_video(source_path=SOURCE_VIDEO, target_path=TARGET_VIDEO, callback=process_frame)
print("Done! Total counted:", line_zone.in_count + line_zone.out_count)

In [ ]:
# Step 4: download the result video to your computer/phone
from google.colab import files
files.download(TARGET_VIDEO)

---
# Part 2: count YOUR own object (eggs, olives, dates, bottles...)

The pretrained model doesn't know eggs, so you train it on a dataset:

1. Go to **universe.roboflow.com**, search your object (e.g. "egg").
2. Open a dataset → **Download Dataset** → format **YOLOv11** → "Show download code". Copy that code into the cell below.
3. Run the training cell (about 20–40 min on the free GPU).
4. In Step 3 above, replace `YOLO("yolo11n.pt")` with `YOLO("runs/detect/train/weights/best.pt")` and set `CLASSES_TO_COUNT = [0]`.

In [ ]:
# Paste your Roboflow download code here, for example:
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# dataset = rf.workspace("...").project("...").version(1).download("yolov11")

# Then train:
# from ultralytics import YOLO
# model = YOLO("yolo11n.pt")
# model.train(data=f"{dataset.location}/data.yaml", epochs=50, imgsz=640)